In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import gc
import io
import json
import time
import zipfile
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import requests
import torch
import torch.nn.functional as F
import transformers
import matplotlib.pyplot as plt

from PIL import Image
from tqdm.auto import tqdm
from transformers import AutoModel, AutoProcessor

In [ ]:
ROOT = Path("/content/drive/MyDrive/MLLM")
DATA = ROOT / "data"
IMAGES = DATA / "images"
OUTPUTS = ROOT / "outputs"

for folder in (DATA, IMAGES, OUTPUTS):
    folder.mkdir(parents=True, exist_ok=True)

CLIP_ID = "openai/clip-vit-base-patch16"
SIGLIP_ID = "google/siglip2-base-patch16-224"
device = "cuda" if torch.cuda.is_available() else "cpu"

clip_en = torch.load(
    OUTPUTS / "clip_en_subset500.pt",
    map_location="cpu",
    weights_only=True,
)

image_ids = clip_en["image_ids"]
print("Device:", device)

if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

print("Transformers:", transformers.__version__)
print("Saved baseline:", clip_en["image_embeddings"].shape)

In [ ]:
manifest_path = DATA / "xm3600_subset500.json"

if not manifest_path.exists():
    response = requests.get(
        "https://google.github.io/crossmodal-3600/web-data/captions.zip",
        timeout=60,
    )
    response.raise_for_status()

    needed = set(image_ids)
    selected = {}

    with zipfile.ZipFile(io.BytesIO(response.content)) as archive:
        with archive.open("captions.jsonl") as source:
            for line in source:
                item = json.loads(line)
                image_id = item["image/key"]

                if image_id in needed:
                    selected[image_id] = {
                        "image_id": image_id,
                        "en": item["en"]["caption"],
                        "bn": item["bn"]["caption"],
                    }

    subset = [selected[image_id] for image_id in image_ids]

    temporary = manifest_path.with_suffix(".json.part")
    temporary.write_text(
        json.dumps(subset, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    temporary.replace(manifest_path)

by_id = {
    item["image_id"]: item
    for item in json.loads(manifest_path.read_text(encoding="utf-8"))
}

subset = [by_id[image_id] for image_id in image_ids]

records = {
    language: [
        {
            "image_id": item["image_id"],
            "caption_index": j,
            "text": caption,
        }
        for item in subset
        for j, caption in enumerate(item[language])
    ]
    for language in ("en", "bn")
}

assert records["en"] == clip_en["caption_records"]

for language in ("en", "bn"):
    print(language, "captions:", len(records[language]))

In [ ]:
def ensure_image(image_id):
    path = IMAGES / f"{image_id}.jpg"

    if path.exists():
        return

    response = requests.get(
        "https://open-images-dataset.s3.amazonaws.com/"
        f"crossmodal-3600/{image_id}.jpg",
        timeout=(15, 60),
    )
    response.raise_for_status()

    with Image.open(io.BytesIO(response.content)) as image:
        image.verify()

    temporary = path.with_suffix(".jpg.part")
    temporary.write_bytes(response.content)
    temporary.replace(path)


missing = [
    image_id for image_id in image_ids
    if not (IMAGES / f"{image_id}.jpg").exists()
]

with ThreadPoolExecutor(max_workers=4) as executor:
    for _ in tqdm(
        executor.map(ensure_image, missing),
        total=len(missing),
        desc="Missing images",
    ):
        pass


def load_rgb(image_id):
    with Image.open(IMAGES / f"{image_id}.jpg") as image:
        return image.convert("RGB")


print("Images ready:", len(image_ids))